In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

In [ ]:
def distance(lon_a, lat_a, lon_b, lat_b):
# Returns distance [m] between pairs of [lon, lat] points [deg]
    
    r_earth = 6371220.0  # Earth's radius [m]

    lat_1 = lat_a * np.pi / 180.  # degrees-to-radians
    lon_1 = lon_a * np.pi / 180.
    lat_2 = lat_b * np.pi / 180.
    lon_2 = lon_b * np.pi / 180.

    dlon = 0.5 * (lon_1 - lon_2)
    dlat = 0.5 * (lat_1 - lat_2)

    return 2.0 * r_earth * np.arcsin(np.sqrt(
        np.sin(dlat) ** 2 +
        np.sin(dlon) ** 2 * np.cos(lat_1) * np.cos(lat_2)
    ) )

In [ ]:
xr.set_options(display_max_rows=64)

In [ ]:
PT_ACTIVE = 0x001  # particle is active

In [ ]:
data = xr.open_dataset("particle_tracking_example.nc")
#print(data)

In [ ]:
# where the particles are released
src_xlon = data["source_locationX"][0, 0]
src_ylat = data["source_locationY"][0, 0]

dmax_m = 0.0  # max distance [m] to any point on any track

num_tracks = int(data.sizes["n"])
for track in range(num_tracks):
    # extract [lon, lat, z] particle positions for a given track
    xlon = data["x"][:, track]
    ylat = data["y"][:, track]

    # keep the parts of the arrays in which particles are active
    flag = data["flag"][:, track]
    okay = flag == PT_ACTIVE

    dist_m = distance(src_xlon, src_ylat, xlon[okay], ylat[okay])

    # keep the max distance so far
    dmax_m = max(dmax_m, np.max(dist_m))

print("Max distance from release [km]:", dmax_m.values / 1000.0)